<a href="https://colab.research.google.com/github/BillJr99/Ursinus-CS357-Fall2026/blob/gh-pages/files/notebooks/Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## What This Notebook Demonstrates

This notebook computes the **attention mechanism** by hand, using tiny handcrafted embeddings, to show how a transformer resolves an ambiguous word ("model" as in *weather model* vs. *fashion model*) by weighting the surrounding context. You will tokenize a sentence, embed each word, compute dot-product attention scores, normalize them with softmax, and blend context vectors into a *contextualized embedding*.

**Course connection:** attention is the core computation inside every LLM that powers the agents we build in this course. When an agent reads a long tool result or a retrieved document and decides which parts matter, it is this same query-key-value weighting doing the work. Understanding it at this scale explains both the power (long-range context) and the cost (the $O(n^2)$ comparison of every token against every other) of agentic context windows.

In [1]:
!pip install numpy
!pip install pandas

In [2]:
# Import necessary libraries
import numpy as np
import pandas as pd

In [3]:
# Function to perform softmax (a function to convert a vector of values into a probability distribution which sums to 1)
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

### YOUR TURN: Sharpening the Softmax

**Before running: what do you expect and why?** If two attention scores are far apart (like 10 and 0) versus close together (like 1 and 0), how will the softmax weights differ? Which case gives a "confident" attention pattern? Write down your prediction before you run the cell below.

In [ ]:
# YOUR TURN: fill in the blanks (___) to compare a "confident" vs. an "uncertain" softmax.
# Fill in: two score vectors -- one with far-apart values, one with close values.

scores_far_apart = np.array([___, ___])   # e.g., very different scores
scores_close = np.array([___, ___])       # e.g., nearly equal scores

print("Softmax of far-apart scores:", softmax(scores_far_apart))
print("Softmax of close scores:   ", softmax(scores_close))

# Check: do the results match your prediction about which one is closer to [0.5, 0.5]?

In [4]:
# Step 1: Tokenization
tokens = ["The", "model", "predicted", "the", "weather", ",", "while", "the", "other", "model", "walked", "the", "runway", "."]
print("Tokens:", tokens)

Tokens: ['The', 'model', 'predicted', 'the', 'weather', ',', 'while', 'the', 'other', 'model', 'walked', 'the', 'runway', '.']


In [5]:
# Step 2: Vectorization
# Assign indices to each token (this is usually done with a tokenizer)
vocab = {word: i for i, word in enumerate(tokens)}
print("\nVocabulary:", vocab)


Vocabulary: {'The': 0, 'model': 9, 'predicted': 2, 'the': 11, 'weather': 4, ',': 5, 'while': 6, 'other': 8, 'walked': 10, 'runway': 12, '.': 13}


# How the Embedding Matrix is Normally Calculated

In practice, the **embedding matrix** is learned systematically during model training.  Here's how it works in modern NLP models like word2vec, GloVe, or transformers.

---

## What is the Embedding Matrix?
The **embedding matrix** is a 2D matrix of shape:

$$
\text{Embedding Matrix} \in \mathbb{R}^{|V| \times d}
$$

- $|V|$ = vocabulary size (number of unique tokens)  
- $d$ = embedding dimension (e.g., 100, 300, 768, etc.)

Each row corresponds to a token in the vocabulary, and the row vector is the embedding for that token.  

Example:  
If `vocab = {"the": 0, "model": 1, "predicted": 2}`,  
then row 0 of the embedding matrix holds the vector for `"the"`, row 1 for `"model"`, etc.

---

## Initialization
Initially, embeddings are **randomly initialized**. Common initialization methods include:
- Uniform distribution: $\mathcal{U}(-\epsilon, \epsilon)$  
- Normal distribution: $\mathcal{N}(0, \sigma^2)$  
- Xavier/Glorot or He initialization (used in deep nets)

So at first, embeddings are just noise.

---

## Training the Embedding Matrix
The embedding vectors are **learned by optimization** (gradient descent) with respect to the model's objective function. Depending on the model:

- **Word2Vec (Skip-gram / CBOW):**  
  Embeddings are trained so that words appearing in similar contexts have similar vectors. Training uses dot products between word and context embeddings, optimized with negative sampling or hierarchical softmax.

- **GloVe:**  
  Embeddings are learned by factorizing the word co-occurrence matrix, so that similarity in embeddings reflects co-occurrence statistics.

- **Transformers (e.g., BERT, GPT):**  
  Embedding matrix is trained jointly with the rest of the network. Tokens are mapped to embeddings, then passed into self-attention layers. The loss (e.g., cross-entropy predicting the next token) propagates back into the embedding matrix, adjusting it.

In all cases, the **backpropagation updates the embedding vectors** just like it updates neural network weights:

$$
E_{t} \leftarrow E_{t} - \eta \cdot \frac{\partial L}{\partial E_{t}}
$$

where:
- $E_t$ = embedding of token $t$  
- $\eta$ = learning rate  
- $L$ = loss

---

## Result
After training:
- Frequent words have **stable embeddings**  
- Semantically related words cluster together (e.g., `"cat"` near `"dog"`)  
- Embeddings can capture analogies (e.g., `"king" - "man" + "woman" ≈ "queen"` in word2vec)

---

In this example, we **manually defined embeddings** (`embedding_matrix = {...}`), but normally they are rows of a single NumPy array (e.g., shape `[vocab_size, d]`).  

- The attention mechanism uses these embeddings, but in reality, the embeddings would come from the trained matrix.  
- Contextualization (via attention) means that even though `"model"` has *one* static embedding row, in a transformer it gets transformed into *different contextualized embeddings* (like the first `"model"` vs. second `"model"` example).

---

The embedding matrix is normally:
1. **Initialized randomly**  
2. **Learned by gradient descent** during training  
3. **Optimized via the model's objective function**  

Each row corresponds to a token's embedding, and the training pushes semantically related tokens closer in vector space.


In [6]:
# Step 3: Embedding (using arbitrary vectors for simplicity)
embedding_matrix = {
    "The": np.array([0.1, 0.2, 0.3]),
    "model": np.array([0.4, 0.1, 0.6]),
    "predicted": np.array([0.7, 0.8, 0.9]),
    "the": np.array([0.1, 0.2, 0.3]),
    "weather": np.array([0.2, 0.5, 0.3]),
    ",": np.array([0.1, 0.0, 0.2]),
    "while": np.array([0.3, 0.6, 0.4]),
    "other": np.array([0.5, 0.1, 0.7]),
    "walked": np.array([0.8, 0.3, 0.5]),
    "runway": np.array([0.6, 0.7, 0.2]),
    ".": np.array([0.0, 0.1, 0.3]),
}

In [7]:
# Step 4: Perform Attention Mechanism
# Calculate attention scores for the word "model" in different contexts

# First "model" (context: predicting the weather)
q1 = embedding_matrix["model"]
k1_1 = embedding_matrix["predicted"]
k1_2 = embedding_matrix["weather"]

attention_scores_1 = np.array([np.dot(q1, k1_1), np.dot(q1, k1_2)])
attention_weights_1 = softmax(attention_scores_1)
print("\nAttention weights for the first 'model':", attention_weights_1)

# Calculate the contextualized embedding for the first "model"
contextualized_embedding_1 = attention_weights_1[0] * embedding_matrix["predicted"] + attention_weights_1[1] * embedding_matrix["weather"]
print("Contextualized embedding for the first 'model':", contextualized_embedding_1)


Attention weights for the first 'model': [0.64336515 0.35663485]
Contextualized embedding for the first 'model': [0.52168257 0.69300954 0.68601909]


In [8]:
# Second "model" (context: walking the runway)
q2 = embedding_matrix["model"]
k2_1 = embedding_matrix["walked"]
k2_2 = embedding_matrix["runway"]

attention_scores_2 = np.array([np.dot(q2, k2_1), np.dot(q2, k2_2)])
attention_weights_2 = softmax(attention_scores_2)
print("\nAttention weights for the second 'model':", attention_weights_2)

# Calculate the contextualized embedding for the second "model"
contextualized_embedding_2 = attention_weights_2[0] * embedding_matrix["walked"] + attention_weights_2[1] * embedding_matrix["runway"]
print("Contextualized embedding for the second 'model':", contextualized_embedding_2)


Attention weights for the second 'model': [0.55477924 0.44522076]
Contextualized embedding for the second 'model': [0.71095585 0.47808831 0.36643377]


### YOUR TURN: Attend from a Different Query Word

**Before running: what do you expect and why?** Suppose the query word is now **"weather"**, attending to the keys **"predicted"** and **"model"**. Based on the embedding vectors defined above, which key do you predict will receive more attention weight? Why? Write your prediction down first; then fill in and run the cell.

In [ ]:
# YOUR TURN: fill in the blanks (___) to compute attention for the query word "weather".
# Follow the same pattern as the cells above.

q3 = embedding_matrix["weather"]
k3_1 = embedding_matrix["predicted"]
k3_2 = embedding_matrix["model"]

# Fill in: the two dot-product attention scores (query dotted with each key)
attention_scores_3 = np.array([___, ___])

# Fill in: convert the raw scores to weights that sum to 1
attention_weights_3 = ___

print("Attention weights for 'weather':", attention_weights_3)

# Fill in: blend the two key/value vectors using the attention weights
contextualized_embedding_3 = ___ * embedding_matrix["predicted"] + ___ * embedding_matrix["model"]
print("Contextualized embedding for 'weather':", contextualized_embedding_3)